# Model Comparison and Selection


This notebook audits the model selection process for the WNBA salary valuation model.

The main goals are:

1. Compare candidate models using the project-defined KPI framework.
2. Confirm that the selected model is consistent with the primary KPI.
3. Check whether the selected model has overfitting or instability concerns.
4. Evaluate final 2025 holdout performance.
5. Inspect player-level prediction errors.

KPI hierarchy used in this notebook:

- **Primary KPI:** RMSE
- **Secondary KPI:** MAPE
- **Business KPI:** CPWS, we do not examine here
- **Supplementary diagnostics:** MAE, R², train-validation gap, `std_test_score` from tuning results, and player-level prediction errors


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score, root_mean_squared_error

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

repo = Path.cwd()

# Walk upward until we find the project root
while not (repo / "data").exists() and repo != repo.parent:
    repo = repo.parent

result_dir = repo / "results" / "Fixed"
art_dir = repo / "artifacts" / "Fixed"

## 1. Load tuning results

The original tuning file is saved as `results/tuning_results.csv`.

For auditing, we create a derived table with additional generalization diagnostics:

- `cv_train_gap = cv_rmse - train_rmse`
- `relative_gap = cv_train_gap / cv_rmse`

These derived columns are not used as the primary selection rule. They are used to audit possible overfitting. `cv_train_gap`: possible overfitting; `relative_gap`: normalized overfitting gap. CV stability is assessed using `std_test_score` from the tuning results.


In [7]:
tuning_path = result_dir / "tuning_results_fixed.csv"
print("tuning_path:", tuning_path)
print("exists:", tuning_path.exists())

tuning_df = pd.read_csv(tuning_path)
tuning_df.head()


tuning_path: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\tuning_results_fixed.csv
exists: True


,params,std_test_score,rank_test_score,model,cv_rmse,train_rmse
0,"{'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 200}",4396.285281,15,RandomForest,40570.148396,25389.322917
1,"{'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 400}",4510.812271,18,RandomForest,40761.529388,25224.394889
2,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}",3466.442185,3,RandomForest,40152.018108,30653.077941
3,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}",3634.664453,9,RandomForest,40324.243405,30498.967899
4,"{'max_depth': None, 'min_samples_leaf': 8, 'n_estimators': 200}",3001.892721,4,RandomForest,40234.190836,35189.149752


In [8]:
# Add audit diagnostics.
tuning_audit = tuning_df.copy()

tuning_audit["cv_train_gap"] = tuning_audit["cv_rmse"] - tuning_audit["train_rmse"]
tuning_audit["relative_gap"] = tuning_audit["cv_train_gap"] / tuning_audit["cv_rmse"]

# Save a derived audit file without overwriting the original tuning results.
audit_dir = result_dir / "audit fixed"
audit_dir.mkdir(parents=True, exist_ok=True)

audit_path = audit_dir  / "tuning_results_audit_fixed.csv"
tuning_audit.to_csv(audit_path, index=False)

print(f"Saved audit tuning results to: {audit_path}")
tuning_audit.head()


Saved audit tuning results to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\audit fixed\tuning_results_audit_fixed.csv


,params,std_test_score,rank_test_score,model,cv_rmse,train_rmse,cv_train_gap,relative_gap
0,"{'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 200}",4396.285281,15,RandomForest,40570.148396,25389.322917,15180.825479,0.374187
1,"{'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 400}",4510.812271,18,RandomForest,40761.529388,25224.394889,15537.134499,0.381172
2,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}",3466.442185,3,RandomForest,40152.018108,30653.077941,9498.940167,0.236574
3,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}",3634.664453,9,RandomForest,40324.243405,30498.967899,9825.275506,0.243657
4,"{'max_depth': None, 'min_samples_leaf': 8, 'n_estimators': 200}",3001.892721,4,RandomForest,40234.190836,35189.149752,5045.041084,0.125392


## 2. Audit tuning-stage model selection

The tuning-stage selection should be driven primarily by cross-validated RMSE, since RMSE is our primary predictive KPI.

However, the audit also checks:

- whether the top-ranked models have very similar CV RMSE values;
- whether the selected model has a large train-validation gap (`cv_train_gap`);
- whether CV performance is stable across chronological folds, using `std_test_score`;


In [9]:
# Sort all candidate configurations by CV RMSE.
top_tuning = tuning_audit.sort_values("cv_rmse").head(10)

top_tuning[[
    "model",
    "params",
    "cv_rmse",
    "train_rmse",
    "cv_train_gap",
    "relative_gap",
    "std_test_score",
    "rank_test_score"
]]


,model,params,cv_rmse,train_rmse,cv_train_gap,relative_gap,std_test_score,rank_test_score
26,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 3, 'n_estimators': 100}",39765.109349,34618.602285,5146.507064,0.129423,4578.369729,1
19,GradientBoosting,"{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 3, 'n_estimators': 200}",39971.438595,33648.763720,6322.674875,0.158180,4624.541111,2
8,RandomForest,"{'max_depth': 6, 'min_samples_leaf': 5, 'n_estimators': 200}",40118.980248,31948.435701,8170.544547,0.203658,3525.834343,1
14,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 200}",40147.145434,30677.784458,9469.360976,0.235866,3468.566655,2
2,RandomForest,"{'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}",40152.018108,30653.077941,9498.940167,0.236574,3466.442185,3
21,GradientBoosting,"{'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 5, 'n_estimators': 200}",40199.344861,33759.826207,6439.518654,0.160190,5008.152532,3
4,RandomForest,"{'max_depth': None, 'min_samples_leaf': 8, 'n_estimators': 200}",40234.190836,35189.149752,5045.041084,0.125392,3001.892721,4
16,RandomForest,"{'max_depth': 10, 'min_samples_leaf': 8, 'n_estimators': 200}",40236.923933,35191.402753,5045.521181,0.125395,2999.451416,5
10,RandomForest,"{'max_depth': 6, 'min_samples_leaf': 8, 'n_estimators': 200}",40267.056104,35507.893190,4759.162915,0.118190,2973.117822,6
28,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 5, 'n_estimators': 100}",40280.071725,34844.154457,5435.917267,0.134953,5284.163353,4


In [10]:
# Best candidate within each model family.
best_by_model = (
    tuning_audit
    .sort_values("cv_rmse")
    .groupby("model", as_index=False)
    .first()
)

best_by_model[[
    "model",
    "params",
    "cv_rmse",
    "train_rmse",
    "cv_train_gap",
    "relative_gap",
    "std_test_score",
    "rank_test_score"
]]


,model,params,cv_rmse,train_rmse,cv_train_gap,relative_gap,std_test_score,rank_test_score
0,GradientBoosting,"{'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 3, 'n_estimators': 100}",39765.109349,34618.602285,5146.507064,0.129423,4578.369729,1
1,RandomForest,"{'max_depth': 6, 'min_samples_leaf': 5, 'n_estimators': 200}",40118.980248,31948.435701,8170.544547,0.203658,3525.834343,1


## Tuning-stage audit notes

The tuning results show that the selected Gradient Boosting model has the lowest cross-validated RMSE among the tuned candidate models. This is consistent with the project KPI framework, since RMSE is used as the primary predictive KPI.

The hyperparameter search was limited to Random Forest and Gradient Boosting. This keeps the optimization process simple and reproducible for a relatively small dataset. Therefore, the selected Gradient Boosting model should be interpreted as the best model within this tuning scope, not necessarily the best possible model across all model families.

The difference between the best Gradient Boosting model and the best Random Forest model is modest. The best Gradient Boosting model has a CV RMSE of 39,765.11, while the best Random Forest model has a CV RMSE of 40,118.98. Gradient Boosting improves CV RMSE by about 353.87, which is small relative to the overall salary scale.

The audit also shows that this RMSE difference should not be over-interpreted. The best Gradient Boosting model has a `std_test_score` of 4,578.37, while the best Random Forest model has a `std_test_score` of 3,525.83. Since these fold-level variability values are much larger than the CV RMSE difference between the two best models, the exact ranking between the top models remains somewhat uncertain.

There is also a generalization trade-off. The best Gradient Boosting model has a smaller train-validation gap, with a `cv_train_gap` of 5,146.51 and a `relative_gap` of 0.129. In comparison, the best Random Forest model has a larger `cv_train_gap` of 8,170.54 and a `relative_gap` of 0.204. **This suggests that Gradient Boosting not only performs slightly better under the primary RMSE KPI, but also shows less overfitting in this tuning audit.**

Overall, selecting Gradient Boosting is reasonable under the RMSE-first rule, while the small performance difference means the final model choice should still be interpreted cautiously.

## 3. Load final 2025 holdout predictions

The file `results/Fixed/tuned_predictions_fixed.csv` contains player-level predictions from the selected tuned model evaluated on the untouched 2025 holdout set.

Expected columns:

- `player`
- `team`
- `year`
- `group`
- `actual`
- `predicted`
- `residual`
- `abs_error`

Here, `residual = actual - predicted`.

The `residual` column helps identify direction:

- positive residual means the model predicted a higher salary than the player actually earned
- negative residual means the model predicted a lower salary than the player actually earned


In [12]:
pred_path = result_dir / "tuned_predictions_fixed.csv"
pred_df = pd.read_csv(pred_path)

pred_df.head()


,player,team,year,group,actual,predicted,residual,abs_error
0,A'ja Wilson,LVA,2025,veteran,200000,227019.917506,27019.917506,27019.917506
1,Aaliyah Edwards,TOT,2025,rookie,74909,69542.147209,-5366.852791,5366.852791
2,Aaliyah Nye,LVA,2025,rookie,69267,73160.608659,3893.608659,3893.608659
3,Aari McDonald,IND,2025,veteran,52333,129520.193440,77187.193440,77187.193440
4,Aerial Powers,TOT,2025,veteran,11924,75257.874048,63333.874048,63333.874048


In [13]:
# Basic data checks.
display(pred_df.shape)
display(pred_df.dtypes)
display(pred_df.isna().sum())

# Recompute residual and absolute error to make sure the saved columns are consistent.
pred_df["residual_check"] = pred_df["actual"] - pred_df["predicted"]
pred_df["abs_error_check"] = pred_df["residual_check"].abs()

pred_df[["actual", "predicted", "residual", "residual_check", "abs_error", "abs_error_check"]].head()


(182, 8)

player           str
team             str
year           int64
group            str
actual         int64
predicted    float64
residual     float64
abs_error    float64
dtype: object

player       0
team         0
year         0
group        0
actual       0
predicted    0
residual     0
abs_error    0
dtype: int64

,actual,predicted,residual,residual_check,abs_error,abs_error_check
0,200000,227019.917506,27019.917506,-27019.917506,27019.917506,27019.917506
1,74909,69542.147209,-5366.852791,5366.852791,5366.852791,5366.852791
2,69267,73160.608659,3893.608659,-3893.608659,3893.608659,3893.608659
3,52333,129520.193440,77187.193440,-77187.193440,77187.193440,77187.193440
4,11924,75257.874048,63333.874048,-63333.874048,63333.874048,63333.874048


## 4. Final 2025 holdout KPI evaluation

This section recomputes the final out-of-sample performance metrics from `results/Fixed/tuned_predictions_fixed.csv`. The tuned model was selected during the cross-validation stage, and the 2025 holdout set is used only as the final test set.

Model performance is interpreted according to the project KPI hierarchy:

1. **RMSE** is the primary predictive KPI.
2. **MAPE** is the secondary scale-sensitive KPI.
3. **MAE** and **R²** are supplementary diagnostics only.

Because the project goal is to compare actual salary with model predicted salary, the holdout metrics evaluate how closely the trained model predicts 2025 player salaries. Player-level residuals are then used to inspect model-relative overpayment or underpayment patterns.


In [18]:
y_true = pred_df["actual"]
y_pred = pred_df["predicted"]

metrics = {
    "model": "GradientBoostingRegressor",
    "train_years": "2021-2024",
    "test_year": 2025,
    "n_train": 633,
    "n_test": len(pred_df),
    "RMSE": root_mean_squared_error(y_true, y_pred),
    "MAE": mean_absolute_error(y_true, y_pred),
    "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    "R2": r2_score(y_true, y_pred),
}

metrics_df = pd.DataFrame([metrics])
metrics_df


,model,train_years,test_year,n_train,n_test,RMSE,MAE,MAPE,R2
0,GradientBoostingRegressor,2021-2024,2025,633,182,39717.19788,30368.066572,0.695309,0.631117


## KPI interpretation

The final tuned Gradient Boosting model achieves a 2025 holdout RMSE of 39,717.20 and an MAE of 30,368.07. Since RMSE is the primary predictive KPI, the main evaluation result is that the model's 2025 holdout prediction error is about $39.7K under RMSE. The MAE means that, on average, the model's player-level salary prediction is off by about $30.4K.

The model also has an R² of 0.631. This means that the model explains a moderate amount of variation in 2025 player salaries compared with a simple mean-prediction baseline. However, R² is only a secondary diagnostic metric. It should not be used as the main model-selection criterion because the project prioritizes salary prediction error measured in dollars.

The MAPE is 0.695, or about 69.5%. This value should be interpreted carefully. MAPE divides each absolute error by the player's actual salary, so players with smaller actual salaries can produce large percentage errors even when the dollar error is not unusually large. Because WNBA salaries include very small hardship or temporary-contract values, aggregate MAPE can be unstable and should not be overemphasized.

Overall, the final Gradient Boosting model is reasonable under the RMSE-first rule. The model provides a usable player-salary valuation benchmark, but RMSE and MAE should be treated as the most reliable evaluation metrics for this project. MAPE should still be reported as a supplementary metric, especially to show percentage-error sensitivity across contract sizes.

## 5. Player-level prediction error analysis

This section identifies the largest player-level prediction errors and uses the residual direction to flag players who appear underpaid or overpaid relative to the model-predicted salary.

Since `residual = predicted - actual`:

1. **largest absolute errors** show the largest prediction mistakes;
2. **largest positive residuals** show players whose predicted salary is much higher than their actual salary, so they appear underpaid relative to the model predicted salary;
3. **smallest residuals** show players whose predicted salary is much lower than their actual salary, so they appear overpaid relative to the model predicted salary.

In [19]:
# Make sure residual and absolute error use the same definition as model_testing.ipynb:
# residual = predicted - actual

pred_df = pred_df.copy()

pred_df["residual"] = pred_df["predicted"] - pred_df["actual"]
pred_df["abs_error"] = pred_df["residual"].abs()

pred_df[[
    "player",
    "team",
    "group",
    "actual",
    "predicted",
    "residual",
    "abs_error"
]].head()

,player,team,group,actual,predicted,residual,abs_error
0,A'ja Wilson,LVA,veteran,200000,227019.917506,27019.917506,27019.917506
1,Aaliyah Edwards,TOT,rookie,74909,69542.147209,-5366.852791,5366.852791
2,Aaliyah Nye,LVA,rookie,69267,73160.608659,3893.608659,3893.608659
3,Aari McDonald,IND,veteran,52333,129520.193440,77187.193440,77187.193440
4,Aerial Powers,TOT,veteran,11924,75257.874048,63333.874048,63333.874048


In [20]:
# Largest absolute prediction errors.
largest_abs_errors = (
    pred_df
    .sort_values("abs_error", ascending=False)
    .head(10)
)

largest_abs_errors[[
    "player",
    "team",
    "group",
    "actual",
    "predicted",
    "residual",
    "abs_error"
]]


,player,team,group,actual,predicted,residual,abs_error
128,Moriah Jefferson,CHI,veteran,145500,31196.673240,-114303.326760,114303.326760
70,Jewell Loyd,LVA,veteran,249032,137885.228522,-111146.771478,111146.771478
152,Sabrina Ionescu,NYL,rookie,222060,112676.942368,-109383.057632,109383.057632
18,Arike Ogunbowale,DAL,unknown,249032,143333.767414,-105698.232586,105698.232586
173,Teaira McCowan,DAL,veteran,201400,96696.534160,-104703.465840,104703.465840
11,Alysha Clark,TOT,veteran,205908,112233.508095,-93674.491905,93674.491905
13,Amy Okonkwo,DAL,veteran,7774,100097.832115,92323.832115,92323.832115
77,Kaila Charles,TOT,veteran,21198,112318.451879,91120.451879,91120.451879
141,Odyssey Sims,TOT,veteran,54622,141170.915845,86548.915845,86548.915845
58,Haley Jones,TOT,veteran,36094,119276.601018,83182.601018,83182.601018


In [21]:
# Most underpaid relative to the model-predicted salary:
# predicted salary is much higher than actual salary. The larger the residual, the more underpaid the player is relative to the model's prediction.
most_underpaid_relative_to_model = (
    pred_df
    .sort_values("residual", ascending=False)
    .head(10)
)

most_underpaid_relative_to_model[[
    "player",
    "team",
    "group",
    "actual",
    "predicted",
    "residual",
    "abs_error"
]]

,player,team,group,actual,predicted,residual,abs_error
13,Amy Okonkwo,DAL,veteran,7774,100097.832115,92323.832115,92323.832115
77,Kaila Charles,TOT,veteran,21198,112318.451879,91120.451879,91120.451879
141,Odyssey Sims,TOT,veteran,54622,141170.915845,86548.915845,86548.915845
58,Haley Jones,TOT,veteran,36094,119276.601018,83182.601018,83182.601018
179,Veronica Burton,GSV,veteran,103831,185805.055296,81974.055296,81974.055296
3,Aari McDonald,IND,veteran,52333,129520.193440,77187.193440,77187.193440
52,Emma Meesseman,NYL,veteran,75694,144695.147959,69001.147959,69001.147959
101,Leonie Fiebich,NYL,unknown,68595,135752.960603,67157.960603,67157.960603
4,Aerial Powers,TOT,veteran,11924,75257.874048,63333.874048,63333.874048
56,Grace Berger,TOT,veteran,17214,79932.668687,62718.668687,62718.668687


In [22]:
# Most overpaid relative to the model-predicted salary:
# predicted salary is much lower than actual salary. The smaller the residual, the more overpaid the player is relative to the model's prediction.
most_overpaid_relative_to_model = (
    pred_df
    .sort_values("residual", ascending=True)
    .head(10)
)

most_overpaid_relative_to_model[[
    "player",
    "team",
    "group",
    "actual",
    "predicted",
    "residual",
    "abs_error"
]]

,player,team,group,actual,predicted,residual,abs_error
128,Moriah Jefferson,CHI,veteran,145500,31196.673240,-114303.326760,114303.326760
70,Jewell Loyd,LVA,veteran,249032,137885.228522,-111146.771478,111146.771478
152,Sabrina Ionescu,NYL,rookie,222060,112676.942368,-109383.057632,109383.057632
18,Arike Ogunbowale,DAL,unknown,249032,143333.767414,-105698.232586,105698.232586
173,Teaira McCowan,DAL,veteran,201400,96696.534160,-104703.465840,104703.465840
11,Alysha Clark,TOT,veteran,205908,112233.508095,-93674.491905,93674.491905
27,Brittney Griner,ATL,veteran,214466,136134.907822,-78331.092178,78331.092178
130,Myisha Hines-Allen,DAL,veteran,203000,126789.560478,-76210.439522,76210.439522
161,Shatori Walker-Kimbrough,ATL,veteran,150000,74763.681218,-75236.318782,75236.318782
45,DiJonai Carrington,TOT,veteran,200000,130289.458623,-69710.541377,69710.541377


## Player-level audit notes

The player-level error analysis identifies where the final Gradient Boosting model differs most from observed 2025 salaries. These results should be interpreted as model-relative valuation signals, not definitive claims that a player is truly underpaid or overpaid.

Using `residual = predicted - actual`, positive residuals indicate players who appear underpaid relative to the model, while negative residuals indicate players who appear overpaid relative to the model.

Large errors are especially important for low-salary players because MAPE divides each error by actual salary. As a result, small salary denominators can create very large percentage errors even when the dollar error is not unusually large.

**The fixed salary de-duplication step ensures that each 2025 player appears only once in the prediction table. Players listed under `TOT` remain useful for player-level valuation, but they should be treated carefully in team-level ROI analysis because their aggregate season records cannot be assigned cleanly to one team.**


## 6. Save audit summary tables

This section saves the key audit tables generated in this notebook.


In [23]:
# Save key audit summary tables generated in this notebook.

audit_dir = result_dir / "audit fixed"
audit_dir.mkdir(parents=True, exist_ok=True)

top_tuning_path = audit_dir / "top_tuning_audit_fixed.csv"
top_tuning.to_csv(top_tuning_path, index=False)

metrics_path = audit_dir / "holdout_metrics_audit_fixed.csv"
metrics_df.to_csv(metrics_path, index=False)

largest_errors_path = audit_dir / "largest_prediction_errors_audit_fixed.csv"
largest_abs_errors.to_csv(largest_errors_path, index=False)

underpaid_path = audit_dir / "most_underpaid_relative_to_model_audit_fixed.csv"
most_underpaid_relative_to_model.to_csv(underpaid_path, index=False)

overpaid_path = audit_dir / "most_overpaid_relative_to_model_audit_fixed.csv"
most_overpaid_relative_to_model.to_csv(overpaid_path, index=False)

print(f"Saved top tuning table to: {top_tuning_path}")
print(f"Saved holdout metrics to: {metrics_path}")
print(f"Saved largest prediction errors to: {largest_errors_path}")
print(f"Saved most underpaid relative to model table to: {underpaid_path}")
print(f"Saved most overpaid relative to model table to: {overpaid_path}")


Saved top tuning table to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\audit fixed\top_tuning_audit_fixed.csv
Saved holdout metrics to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\audit fixed\holdout_metrics_audit_fixed.csv
Saved largest prediction errors to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\audit fixed\largest_prediction_errors_audit_fixed.csv
Saved most underpaid relative to model table to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\audit fixed\most_underpaid_relative_to_model_audit_fixed.csv
Saved most overpaid relative to model table to: c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\results\Fixed\audit fixed\most_overpaid_relative_to_model_audit_fixed.csv


## 7. Final model comparison conclusion

The final selected model is the tuned Gradient Boosting model.

Key audit conclusions:

- **Model selection:** Gradient Boosting is the final tuned model under the RMSE-first rule. It achieved the lowest cross-validated RMSE among the tuned candidate models.

- **Baseline comparison:** The broader baseline comparison included Random Forest, Ridge, Linear Regression, and Dummy Regressor. These results are saved in `baseline_evaluation.csv` and provide context for the tuned model results.

- **Tuning scope:** The tuning stage focused only on Random Forest and Gradient Boosting. Therefore, the final tuned model should be interpreted as the best model within this tuning scope, not necessarily the best possible model across all model families.

- **CV variability:** The best Gradient Boosting model only improves CV RMSE over the best Random Forest model by about 353.87. This difference is small compared with the fold-level `std_test_score` values, so the exact ranking between the top tuned models should not be over-interpreted.

- **Generalization gap:** Gradient Boosting has a smaller train-validation gap than Random Forest, suggesting less overfitting in the tuning audit.

- **2025 holdout performance:** On the 2025 holdout set, the final Gradient Boosting model achieves an RMSE of about $39.7K and an MAE of about $30.4K, which provides a reasonable player-level salary prediction benchmark.

- **MAPE interpretation:** The aggregate MAPE is sensitive to small actual salary values, especially for hardship or temporary-contract players. **Therefore, the final evaluation should emphasize RMSE, MAE, and player-level residual analysis rather than relying heavily on MAPE.**